# 20 OTM Wikipedia Enrichment

Attach Wikipedia titles and pageview-based popularity features to the final OpenTripMap POI base.

In [1]:
import re
import time
from urllib.parse import unquote, urlparse

import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)


In [2]:
INPUT_PATH = "../data/processed/otm_pois_base_final.csv"
OUTPUT_PATH = "../data/processed/otm_pois_with_wikipedia.csv"
UNMATCHED_OUTPUT_PATH = "../data/processed/otm_pois_without_wikipedia.csv"

WIKIPEDIA_API_URL = "https://en.wikipedia.org/w/api.php"
WIKIMEDIA_PAGEVIEWS_URL = "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/{project}/all-access/user/{title}/daily/{start}/{end}"
USER_AGENT = "tourism-crowd-forecasting-research/1.0"

START_DATE = "20250101"
END_DATE = "20251231"
MAX_POIS = None  # set to an integer for testing

poi_df = pd.read_csv(INPUT_PATH)
if MAX_POIS is not None:
    poi_df = poi_df.head(MAX_POIS).copy()

print("POIs loaded:", poi_df.shape)


POIs loaded: (327, 18)


## Helpers

In [3]:
def extract_title_from_url(url):
    if pd.isna(url) or not str(url).strip():
        return None
    parsed = urlparse(str(url))
    path = parsed.path or ""
    if "/wiki/" not in path:
        return None
    title = path.split("/wiki/", 1)[1]
    title = unquote(title)
    return title.replace("_", " ").strip() or None


def normalize_title_for_pageviews(title):
    return str(title).replace(" ", "_")


def search_wikipedia(query, limit=5, max_retries=5):
    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "utf8": 1,
        "format": "json",
        "srlimit": limit,
    }

    headers = {"User-Agent": USER_AGENT}

    for attempt in range(max_retries):
        response = requests.get(WIKIPEDIA_API_URL, params=params, headers=headers, timeout=20)
        if response.status_code == 200:
            return response.json().get("query", {}).get("search", [])
        if response.status_code == 429:
            time.sleep(2 ** attempt)
            continue
        response.raise_for_status()

    return []


def token_set(text):
    text = "" if pd.isna(text) else str(text).casefold()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return {t for t in text.split() if len(t) > 2}


def title_match_score(query, title):
    q = token_set(query)
    t = token_set(title)
    if not q or not t:
        return 0.0
    return len(q & t) / len(q | t)


def choose_best_search_result(query, results):
    if not results:
        return None
    scored = []
    for result in results:
        title = result.get("title")
        score = title_match_score(query, title)
        scored.append((score, result))
    scored.sort(key=lambda x: x[0], reverse=True)
    best_score, best = scored[0]
    if best_score < 0.20:
        return None
    return best


def get_pageviews(title, start_date, end_date, max_retries=5):
    article = normalize_title_for_pageviews(title)
    url = WIKIMEDIA_PAGEVIEWS_URL.format(
        project="en.wikipedia.org",
        title=article,
        start=start_date,
        end=end_date,
    )
    headers = {"User-Agent": USER_AGENT}

    for attempt in range(max_retries):
        response = requests.get(url, headers=headers, timeout=20)
        if response.status_code == 200:
            return response.json().get("items", [])
        if response.status_code in (404, 429):
            time.sleep(2 ** attempt)
            if response.status_code == 404:
                return []
            continue
        response.raise_for_status()

    return []


## Match titles

In [4]:
poi_df["wiki_title"] = poi_df["wikipedia_url"].apply(extract_title_from_url)
poi_df["wiki_match_method"] = np.where(poi_df["wiki_title"].notna(), "from_url", "")
poi_df["wiki_match_score"] = np.where(poi_df["wiki_title"].notna(), 1.0, np.nan)

search_mask = poi_df["wiki_title"].isna()
search_queries = poi_df.loc[search_mask, ["display_name_en"]].copy()

for idx, row in search_queries.iterrows():
    query = row["display_name_en"]
    try:
        results = search_wikipedia(query)
        best = choose_best_search_result(query, results)
        if best is not None:
            poi_df.at[idx, "wiki_title"] = best.get("title")
            poi_df.at[idx, "wiki_match_method"] = "search"
            poi_df.at[idx, "wiki_match_score"] = title_match_score(query, best.get("title"))
        time.sleep(0.1)
    except Exception as exc:
        poi_df.at[idx, "wiki_match_method"] = f"search_error: {type(exc).__name__}"

poi_df[["display_name_en", "wiki_title", "wiki_match_method", "wiki_match_score"]].head(30)


,display_name_en,wiki_title,wiki_match_method,wiki_match_score
0,Chora Mosque / Kariye Museum,Chora Church,from_url,1.000000
1,Hagia Sophia,Hagia Sophia,from_url,1.000000
2,Serpent Column,Serpent Column,from_url,1.000000
3,Süleymaniye Mosque,Süleymaniye Mosque,from_url,1.000000
4,The Blue Mosque,Sultan Ahmed Mosque,from_url,1.000000
5,Tomb of Sultan Ahmet,Sultan Ahmed Mosque,from_url,1.000000
6,Yıldız Palace,Yıldız Palace,from_url,1.000000
7,15 July coup monument (Istanbul),Timeline of Istanbul,search,0.200000
8,Abbas Ağa Fountain,NaN,,NaN
9,Adam Mickiewicz Museum,Adam Mickiewicz Museum,search,1.000000


## Pull pageviews

In [5]:
poi_df["wiki_has_page"] = poi_df["wiki_title"].notna().astype(int)
poi_df["wiki_pageviews_total"] = 0.0
poi_df["wiki_pageviews_avg"] = 0.0
poi_df["wiki_pageviews_max"] = 0.0
poi_df["wiki_days_observed"] = 0

for idx, row in poi_df.loc[poi_df["wiki_title"].notna()].iterrows():
    title = row["wiki_title"]
    try:
        items = get_pageviews(title, START_DATE, END_DATE)
        views = [item.get("views", 0) for item in items]
        if views:
            poi_df.at[idx, "wiki_pageviews_total"] = float(sum(views))
            poi_df.at[idx, "wiki_pageviews_avg"] = float(sum(views) / len(views))
            poi_df.at[idx, "wiki_pageviews_max"] = float(max(views))
            poi_df.at[idx, "wiki_days_observed"] = int(len(views))
        time.sleep(0.1)
    except Exception as exc:
        if not str(poi_df.at[idx, "wiki_match_method"]).startswith("search_error"):
            poi_df.at[idx, "wiki_match_method"] = f"pageviews_error: {type(exc).__name__}"

max_total = poi_df["wiki_pageviews_total"].max()
poi_df["wiki_popularity_score"] = np.where(max_total > 0, poi_df["wiki_pageviews_total"] / max_total, 0.0)

poi_df[["display_name_en", "wiki_title", "wiki_pageviews_total", "wiki_popularity_score"]].sort_values("wiki_pageviews_total", ascending=False).head(30)


,display_name_en,wiki_title,wiki_pageviews_total,wiki_popularity_score
70,"Edirnekapı, Istanbul",Istanbul,1617259.0,1.000000
105,Istanbul City Wall,Istanbul,1617259.0,1.000000
263,Istanbul Radio House,Istanbul,1617259.0,1.000000
1,Hagia Sophia,Hagia Sophia,1300243.0,0.803979
43,Byzantium,Byzantium,296282.0,0.183200
200,Topkapı Palace,Topkapı Palace,268748.0,0.166175
123,Mausoleum of Mahmud II,Mahmud II,248567.0,0.153696
69,Ecumenical Patriarchate of Constantinople,Ecumenical Patriarchate of Constantinople,217500.0,0.134487
207,Walls of Constantinople,Walls of Constantinople,209372.0,0.129461
206,Vodafone Park,Vodafone Idea,185553.0,0.114733


## Missing-page handling

In [6]:
unmatched_df = poi_df.loc[poi_df["wiki_has_page"] == 0].copy()

print("Total POIs:", len(poi_df))
print("With wiki page:", int(poi_df["wiki_has_page"].sum()))
print("Without wiki page:", len(unmatched_df))

unmatched_df[["display_name_en", "category_clean", "wikidata", "query_area"]].head(30)


Total POIs: 327
With wiki page: 254
Without wiki page: 73


,display_name_en,category_clean,wikidata,query_area
8,Abbas Ağa Fountain,historic,Q6063427,Uskudar Waterfront
13,Alfred Heilbronn Botanical Garden,attraction,Q55594541,Suleymaniye
14,All Saints Moda English Church,religious,Q20476315,Kadikoy Historic Center
22,Aya Pandeleimon Rum Ortodoks Kilisesi,religious,Q97403030,Beylerbeyi Palace
24,Ayios Fokas Church,religious,Q59238241,Ortakoy
31,Balat Yanbol Sinagogu,religious,Q3409182,Balat / Fener
35,Beth Yaakov Sinagogu,religious,Q4897165,Beylerbeyi Palace
39,Bezmialem Valide Sultan Camii,religious,Q2086068,Dolmabahce
67,Dostlar Tiyatrosu,attraction,Q6100278,Suleymaniye
93,Halid Ağa Çeşmesi,historic,Q97368741,Kadikoy Historic Center


## Save outputs

In [7]:
poi_df.to_csv(OUTPUT_PATH, index=False)
unmatched_df.to_csv(UNMATCHED_OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH, poi_df.shape)
print("Saved:", UNMATCHED_OUTPUT_PATH, unmatched_df.shape)


Saved: ../data/processed/otm_pois_with_wikipedia.csv (327, 27)
Saved: ../data/processed/otm_pois_without_wikipedia.csv (73, 27)


## Note on POIs without Wikipedia pages

POIs without a Wikipedia page are kept in the dataset. Their wiki flags stay at 0 and pageview features stay at 0. They should remain usable in later modeling with non-Wikipedia features such as `rate`, `category_clean`, and location fields.